# Legacy Block Re-Synchronization

This notebook allows you to **re-synchronize a legacy block** (one analyzed with the previous pipeline) to the **new paradigm** without re-running heavy analysis steps (jitter correction, ellipse fitting, degree conversion).

## What it does

1. **Load a legacy block** (recording folder that already has `left_eye_data.csv`, `right_eye_data.csv`, and optionally `final_sync_df.csv` from the old pipeline).
2. **Re-parse Open Ephys events** (including falling edges, e.g. `LED_driver_fall`) using the same logic as `block_synchronization.ipynb`.
3. **Rebuild synchronization only**: simple sync → optional manual shift → merge onto 60 Hz arena grid → new `final_sync_df`.
4. **Remap legacy eye data** to the new sync: for each row in the new `final_sync_df`, look up the legacy `left_eye_data` / `right_eye_data` by `eye_frame` and copy over the **corrected degree-based (and other) columns**, while setting **new** `OE_timestamp` and `ms_axis` from the new sync.
5. **Export**: Write new `final_sync_df`. For eye data, you **choose** which legacy CSV files to use (they can have any name). The current file at the output path (if any) is renamed to `***_legacy.csv`; the newly synced data is written under the **original filename** you chose.

## Prerequisite

**Run `block_synchronization.ipynb` in the same kernel** from the top through the cell that defines:
- `simple_sync_build`, `build_final_sync_df_merge_nearest`
- `export_final_sync_df`, `load_final_sync_df`

That is approximately **cells 1–3** of `block_synchronization.ipynb`. Then run this notebook. Alternatively, run that notebook first (same kernel), then switch to this notebook.

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

from eye_tracking_system_tools.preprocessing import utility_functions as uf

# Sync functions (build_final_sync_df_merge_nearest, simple_sync_build, export_final_sync_df, etc.)
# must be defined in this kernel. Run block_synchronization.ipynb cells 1-3 first.
try:
    build_final_sync_df_merge_nearest
except NameError:
    raise NameError(
        "Sync functions not found. Run block_synchronization.ipynb cells 1-3 "
        "(through build_final_sync_df_merge_nearest and export_final_sync_df) in this kernel first."
    )

## 1. Block setup and channel mapping

Set `experiment_path`, `block_numbers`, and `animal`. Configure `channeldict` for your paradigm (same as in `block_synchronization.ipynb`).

In [ ]:
experiment_path = Path(r"D:\sample_data_for_eye_repo")  # adjust to your recording root
block_numbers = [15]  # legacy block(s) to re-sync
animal = 'PV_106'
bad_blocks = []

block_collection = uf.block_generator(
    block_numbers=block_numbers,
    experiment_path=experiment_path,
    animal=animal,
    bad_blocks=bad_blocks,

)

# Channel mapping (must match your Open Ephys TTL wiring)
for block in block_collection:
    if block.animal_call == 'PV_208':
        block.channeldict = {1: 'LED_driver', 7: 'L_eye_TTL', 2: 'Arena_TTL', 8: 'R_eye_TTL'}
    elif block.animal_call == "TE_21":
        block.channeldict = {1: 'Arena_TTL', 4: 'LED_driver', 5: 'R_eye_TTL', 8: 'L_eye_TTL'}
    elif block.animal_call == "PV_106":
        block.channeldict = {1: 'LED_driver', 7: 'L_eye_TTL', 2: 'Arena_TTL', 8: 'R_eye_TTL'}
    else:
        block.channeldict = None  # will use manual TTL selector in parse_open_ephys_events

block_dict = {str(b.block_num): b for b in block_collection}
block = block_collection[0]  # work with first block
print(f"Block {block.block_num}, analysis_path = {block.analysis_path}")

## 2. Re-parse Open Ephys events and prepare for sync

This re-reads OE events (including falling edges) and builds brightness vectors so we can run the same sync pipeline as in `block_synchronization.ipynb`. **No jitter correction or ellipse fitting** is run.

In [ ]:
block.handle_eye_videos()
block.parse_open_ephys_events()  # re-parses OE events, adds falling edges (e.g. LED_driver_fall)
block.handle_arena_files()
block.get_eye_brightness_vectors()

print(f"oe_events columns: {list(block.oe_events.columns)}")
if 'LED_driver_fall' in block.oe_events.columns:
    n_fall = block.oe_events['LED_driver_fall'].notna().sum()
    print(f"LED_driver_fall: {n_fall} falling edges")

## 3. Build new synchronization (same as block_synchronization.ipynb)

Optional: run `build_arena_grid_df` if you use it in the main notebook. Then run simple sync, optional manual shift, and merge onto the 60 Hz grid.

In [ ]:
# Simple sync (anchor at first TTL, internal timing)
dfL, dfR = simple_sync_build(block, cov_warn=0.05, export=False)

# Optional: inspect alignment and apply manual shift (same as block_synchronization)
# plot_simple_sync_bokeh(block, df_left=dfL, df_right=dfR, show_led=True, to_browser=True)
# Then set shift values from the sliders. shift_eye_df_by_index is defined in block_synchronization.ipynb.
shift_left = 0
shift_right = 0

if shift_left != 0 or shift_right != 0:
    dfL_shifted = shift_eye_df_by_index(dfL, shift_left)
    dfR_shifted = shift_eye_df_by_index(dfR, shift_right)
else:
    dfL_shifted = dfL
    dfR_shifted = dfR

In [ ]:
# Merge onto 60 Hz arena grid → new final_sync_df
final_df = build_final_sync_df_merge_nearest(
    block, dfL_shifted, dfR_shifted,
    target_fps=60.0,
    tol_frac=0.90,
    pre_shift_left=0,
    pre_shift_right=0,
    export_csv=False,  # we will export after remapping eye data
)
block.final_sync_df = final_df
print(f"New final_sync_df: {len(final_df)} rows")

## 4. Select legacy eye data files

Set **`legacy_left_path`** and **`legacy_right_path`** to the CSV files that contain the corrected eye data you want to re-sync (any name/location). The next section will remap them; on export, the current file at the output path will be renamed to `stem_legacy.csv` and the new synced data will be written under the **original filename** you chose.

In [ ]:
# List CSV files in the block's analysis folder (for convenience)
ap = Path(block.analysis_path)
csvs = sorted(ap.glob("*.csv"))
print("CSVs in block.analysis_path:")
for f in csvs:
    print(f"  {f.name}")

# --- Set these to the left/right eye data files you want to use (edit paths) ---
legacy_left_path = Path(ap / "left_eye_data.csv")   # e.g. ap / "left_eye_data_degrees_raw_verified.csv"
legacy_right_path = Path(ap / "right_eye_data.csv")   # e.g. ap / "right_eye_data_degrees_raw_verified.csv"
# Paths can be anywhere; output will be written to block.analysis_path with the same filename.

## 5. Load legacy eye data and remap to new sync

**Manually set** `legacy_left_path` and `legacy_right_path` to the CSV files that contain the corrected degree-based (or pixel) eye data you want to re-sync. They can be anywhere and have any name (e.g. `left_eye_data_degrees_raw_verified.csv`). Run the cell above to list CSVs in the block's analysis folder if needed. Then remap: match by `eye_frame`, keep all data columns, overwrite only `OE_timestamp` and `ms_axis` from the new sync.

In [ ]:
def remap_eye_data_to_new_sync(block, final_sync_df, legacy_left_path, legacy_right_path):
    """
    Build new left_eye_data and right_eye_data from the new final_sync_df and legacy CSVs.
    For each row in final_sync_df we have (Arena_TTL, L_eye_frame, R_eye_frame).
    Look up legacy row by eye_frame, copy data columns (center_x, center_y, etc.),
    set OE_timestamp = Arena_TTL and ms_axis = Arena_TTL / (sample_rate/1000).
    """
    fs = getattr(block, 'sample_rate', None) or float(block.get_sample_rate())
    ap = Path(block.analysis_path)

    leg_L = pd.read_csv(legacy_left_path, index_col=0, engine='python')
    leg_R = pd.read_csv(legacy_right_path, index_col=0, engine='python')

    # Columns we set from new sync (OE timebase)
    sync_cols = ['OE_timestamp', 'ms_axis', 'eye_frame']
    # Data columns to copy from legacy (everything except sync-related)
    def data_columns(df, frame_col='eye_frame'):
        skip = {'OE_timestamp', 'ms_axis', frame_col}
        return [c for c in df.columns if c not in skip]

    # ---- Left: key by L_eye_frame ----
    frame_col_L = 'L_eye_frame'
    sf = final_sync_df[['Arena_TTL', frame_col_L]].copy()
    sf = sf.rename(columns={frame_col_L: 'eye_frame'})
    sf['OE_timestamp'] = sf['Arena_TTL']
    sf['ms_axis'] = sf['Arena_TTL'] / (fs / 1000)
    # Drop rows where no eye frame is assigned
    sf_valid = sf.dropna(subset=['eye_frame'])
    sf_valid['eye_frame'] = sf_valid['eye_frame'].astype(int)

    # Legacy may use 'eye_frame' or 'L_eye_frame' for left
    frame_key_L = 'eye_frame' if 'eye_frame' in leg_L.columns else ('L_eye_frame' if 'L_eye_frame' in leg_L.columns else None)
    if frame_key_L is None:
        raise ValueError("Legacy left_eye_data must have 'eye_frame' or 'L_eye_frame' column.")
    leg_L_trim = leg_L.copy()
    if frame_key_L != 'eye_frame':
        leg_L_trim = leg_L_trim.rename(columns={frame_key_L: 'eye_frame'})
    leg_L_trim = leg_L_trim[['eye_frame'] + data_columns(leg_L_trim)]

    left_merged = sf_valid.merge(leg_L_trim, on='eye_frame', how='left', suffixes=('', '_leg'))
    # Drop any duplicate columns from merge
    left_merged = left_merged[[c for c in left_merged.columns if not c.endswith('_leg')]]
    left_merged = left_merged.drop(columns=['Arena_TTL'], errors='ignore')
    new_left = left_merged

    # ---- Right: key by R_eye_frame ----
    frame_col_R = 'R_eye_frame'
    sfR = final_sync_df[['Arena_TTL', frame_col_R]].copy()
    sfR = sfR.rename(columns={frame_col_R: 'eye_frame'})
    sfR['OE_timestamp'] = sfR['Arena_TTL']
    sfR['ms_axis'] = sfR['Arena_TTL'] / (fs / 1000)
    sfR_valid = sfR.dropna(subset=['eye_frame'])
    sfR_valid['eye_frame'] = sfR_valid['eye_frame'].astype(int)

    # Legacy may use 'eye_frame' or 'R_eye_frame' for right
    frame_key_R = 'eye_frame' if 'eye_frame' in leg_R.columns else ('R_eye_frame' if 'R_eye_frame' in leg_R.columns else None)
    if frame_key_R is None:
        raise ValueError("Legacy right_eye_data must have 'eye_frame' or 'R_eye_frame' column.")
    leg_R_trim = leg_R.copy()
    if frame_key_R != 'eye_frame':
        leg_R_trim = leg_R_trim.rename(columns={frame_key_R: 'eye_frame'})
    leg_R_trim = leg_R_trim[['eye_frame'] + data_columns(leg_R_trim)]

    right_merged = sfR_valid.merge(leg_R_trim, on='eye_frame', how='left', suffixes=('', '_leg'))
    right_merged = right_merged[[c for c in right_merged.columns if not c.endswith('_leg')]]
    right_merged = right_merged.drop(columns=['Arena_TTL'], errors='ignore')
    new_right = right_merged

    return new_left, new_right


# Paths set in the cell above ("Select legacy eye data files")
if not legacy_left_path.exists() or not legacy_right_path.exists():
    raise FileNotFoundError(
        f"Legacy eye data files not found. Left: {legacy_left_path}\nRight: {legacy_right_path}"
    )

new_left_eye_data, new_right_eye_data = remap_eye_data_to_new_sync(
    block, block.final_sync_df, legacy_left_path, legacy_right_path
)
block.left_eye_data = new_left_eye_data
block.right_eye_data = new_right_eye_data
print(f"Remapped left_eye_data: {len(new_left_eye_data)} rows")
print(f"Remapped right_eye_data: {len(new_right_eye_data)} rows")
print(f"Left columns: {list(new_left_eye_data.columns)}")

## 6. Export updated CSVs

Writes new `final_sync_df`. For eye data: the **current** file at the output path (same filename as the legacy file you chose) is renamed to `stem_legacy.csv`; the newly synced data is written to the **original** filename (e.g. `left_eye_data_degrees_raw_verified.csv`). Output is always under `block.analysis_path`.

In [ ]:
# Export new final_sync_df
export_final_sync_df(block, final_df=block.final_sync_df, overwrite=True)

# Eye data: backup existing file to *_legacy.csv, then write new synced data to original filename
def export_eye_data_with_legacy_backup(block, legacy_path, df, label='left'):
    legacy_path = Path(legacy_path)
    out_path = Path(block.analysis_path) / legacy_path.name
    if out_path.exists():
        legacy_backup = out_path.parent / (out_path.stem + '_legacy' + out_path.suffix)
        out_path.rename(legacy_backup)
        print(f"  [{label}] Renamed {out_path.name} -> {legacy_backup.name}")
    df.to_csv(out_path)
    print(f"  [{label}] Wrote {out_path.name}")

export_eye_data_with_legacy_backup(block, legacy_left_path, block.left_eye_data, 'left')
export_eye_data_with_legacy_backup(block, legacy_right_path, block.right_eye_data, 'right')
print(f"[OK] All files written to {block.analysis_path}")

## Optional: Sanity plot

If `sanity_plot_final_df` is available from `block_synchronization.ipynb`, you can verify the new sync against LED events.

In [ ]:
# Uncomment if sanity_plot_final_df was run in block_synchronization:
# sanity_plot_final_df(block.final_sync_df, block.sample_rate, show_led=True, block=block)